<a href="https://colab.research.google.com/github/catafest/colab_google/blob/master/catafest_073.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Testing ``ipywidgets``.

*Ipywidgets are interactive HTML widgets for Jupyter notebooks and the IPython kernel that allow users to create GUIs (sliders, text boxes, checkboxes) to visualize and control data in real time. They enable dynamic data exploration and parameter tuning, with over 30 built-in, customizable controls for building interactive dashboards directly in Python code.*

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

uploaded_text_files = {}   # filename -> text content

# Create tab container
tab = widgets.Tab()
tab.children = [widgets.Output(), widgets.Output(), widgets.Output()]
tab.set_title(0, "GUI")
tab.set_title(1, "Files")
tab.set_title(2, "Preview")

gui_tab = tab.children[0]
files_tab = tab.children[1]
preview_tab = tab.children[2]

preview_output = widgets.Output()


# ============================================================
# TAB 1 — GUI ELEMENTS + OPEN/SAVE TEXT FILES
# ============================================================
with gui_tab:
    print("GUI elements and file operations")

    # Basic widgets
    btn = widgets.Button(description="Button")
    txt = widgets.Text(description="Text")
    txt_area = widgets.Textarea(description="Textarea")
    slider = widgets.IntSlider(description="Slider", min=0, max=100)
    dropdown = widgets.Dropdown(description="Dropdown", options=["A","B","C"])
    checkbox = widgets.Checkbox(description="Check")
    toggle = widgets.ToggleButton(description="Toggle")

    gui_out = widgets.Output()

    def on_btn_click(b):
        with gui_out:
            gui_out.clear_output()
            print("Button clicked")

    btn.on_click(on_btn_click)

    # OPEN TEXT FILE
    open_btn = widgets.Button(description="Open text file")
    open_out = widgets.Output()

    def open_file(b):
        with open_out:
            open_out.clear_output()
            print("Choose a text file")
            up = files.upload()
            for name, data in up.items():
                try:
                    text = data.decode("utf-8")
                    uploaded_text_files[name] = text
                    print("Loaded:", name)
                except:
                    print("Invalid text file:", name)

    open_btn.on_click(open_file)

    # SAVE TEXT FILE
    save_btn = widgets.Button(description="Save text")
    save_text = widgets.Textarea(description="Content")
    save_out = widgets.Output()

    def save_file(b):
        with save_out:
            save_out.clear_output()
            content = save_text.value
            if not content.strip():
                print("No text to save")
                return
            filename = "saved_text.txt"
            with open(filename, "w") as f:
                f.write(content)
            files.download(filename)
            print("Saved:", filename)

    save_btn.on_click(save_file)

    display(
        btn, txt, txt_area, slider, dropdown, checkbox, toggle,
        gui_out,
        widgets.HTML("<hr>"),
        open_btn, open_out,
        widgets.HTML("<hr>"),
        save_text, save_btn, save_out
    )


# ============================================================
# TAB 2 — LIST TEXT FILES AND CLICK TO PREVIEW IN TAB 3
# ============================================================
with files_tab:
    print("Loaded text files")

    list_out = widgets.Output()

    def refresh_list():
        with list_out:
            list_out.clear_output()
            if not uploaded_text_files:
                print("No text files loaded")
            else:
                for name in uploaded_text_files.keys():
                    btn = widgets.Button(description=name, layout=widgets.Layout(width="300px"))

                    def make_handler(n):
                        def handler(b):
                            show_preview(n)
                            tab.selected_index = 2  # switch to Preview tab
                        return handler

                    btn.on_click(make_handler(name))
                    display(btn)

    refresh_btn = widgets.Button(description="Refresh")
    refresh_btn.on_click(lambda b: refresh_list())

    display(refresh_btn, list_out)


# ============================================================
# TAB 3 — PREVIEW SELECTED TEXT FILE WITH EXPLANATIONS
# ============================================================
with preview_tab:
    display(preview_output)

def show_preview(filename):
    preview_output.clear_output()
    text = uploaded_text_files[filename]

    explanation = (
        "File preview\n"
        "• This panel shows the full text content of the selected file.\n"
        "• The file was loaded from the upload dialog in Tab 1.\n"
        "• Clicking a filename in Tab 2 automatically switches to this tab.\n"
        "• The text below is displayed exactly as stored.\n"
    )

    with preview_output:
        print("=== File:", filename, "===\n")
        print(explanation)
        print("--- File Content ---")
        print(text)

display(tab)